In [71]:
from fudgeo.geopkg import GeoPackage
from pathlib import Path
from shapely.geometry import Point, Polygon
import osmnx as ox
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx

In [72]:
path = Path('../data/RS_setores_CD2022.gpkg')
if path.is_file():
    gpkg = GeoPackage(path)
else:
    raise FileNotFoundError

In [73]:
print("Feature Classes:", list(gpkg.feature_classes))
print("Tables:", list(gpkg.tables))

Feature Classes: ['RS_setores_CD2022']
Tables: []


In [74]:
cursor = gpkg.connection.execute("""
    SELECT name
    FROM sqlite_schema
    WHERE type ='table' AND 
    name NOT LIKE 'sqlite_%';
""")
features = cursor.fetchall()
print(features)


# for shape, name, status in features:
#     # 'shape' is automatically converted to a fudgeo geometry object
#     print(f"Name: {name} | Status: {status} | Geometry: {shape}")

[('gpkg_spatial_ref_sys',), ('gpkg_contents',), ('gpkg_geometry_columns',), ('gpkg_tile_matrix_set',), ('gpkg_tile_matrix',), ('RS_setores_CD2022',), ('gpkg_extensions',), ('rtree_RS_setores_CD2022_geom',), ('rtree_RS_setores_CD2022_geom_rowid',), ('rtree_RS_setores_CD2022_geom_node',), ('rtree_RS_setores_CD2022_geom_parent',)]


In [75]:
cursor = gpkg.connection.execute("""
PRAGMA table_info(RS_setores_CD2022);
""")
features = cursor.fetchall()
print(features)

[(0, 'id', 'INTEGER', 1, None, 1), (1, 'geom', 'POLYGON', 0, None, 0), (2, 'CD_SETOR', 'TEXT(15)', 0, None, 0), (3, 'SITUACAO', 'TEXT(6)', 0, None, 0), (4, 'CD_SIT', 'TEXT(1)', 0, None, 0), (5, 'CD_TIPO', 'TEXT(1)', 0, None, 0), (6, 'AREA_KM2', 'REAL', 0, None, 0), (7, 'CD_REGIAO', 'TEXT(1)', 0, None, 0), (8, 'NM_REGIAO', 'TEXT(15)', 0, None, 0), (9, 'CD_UF', 'TEXT(2)', 0, None, 0), (10, 'NM_UF', 'TEXT(20)', 0, None, 0), (11, 'CD_MUN', 'TEXT(7)', 0, None, 0), (12, 'NM_MUN', 'TEXT(50)', 0, None, 0), (13, 'CD_DIST', 'TEXT(9)', 0, None, 0), (14, 'NM_DIST', 'TEXT(50)', 0, None, 0), (15, 'CD_SUBDIST', 'TEXT(11)', 0, None, 0), (16, 'NM_SUBDIST', 'TEXT(50)', 0, None, 0), (17, 'CD_BAIRRO', 'TEXT(13)', 0, None, 0), (18, 'NM_BAIRRO', 'TEXT(254)', 0, None, 0), (19, 'CD_NU', 'TEXT(10)', 0, None, 0), (20, 'NM_NU', 'TEXT(254)', 0, None, 0), (21, 'CD_FCU', 'TEXT(11)', 0, None, 0), (22, 'NM_FCU', 'TEXT(254)', 0, None, 0), (23, 'CD_AGLOM', 'TEXT(12)', 0, None, 0), (24, 'NM_AGLOM', 'TEXT(254)', 0, None,

In [76]:
fc = gpkg.feature_classes['RS_setores_CD2022']
cursor = gpkg.connection.execute(f"""
select {fc.geometry_column_name} from {fc.name};
""")
one = cursor.fetchone()
polygon_shape = one[0]  # fudgeo Polygon object

# Access coordinate rings
print("Exterior Ring Coordinates:", polygon_shape.rings[0].coordinates)

Exterior Ring Coordinates: [[-52.6275161 -32.1502212]
 [-52.6281617 -32.1508949]
 [-52.6289383 -32.1516153]
 ...
 [-52.6260004 -32.1490611]
 [-52.6268331 -32.1496786]
 [-52.6275161 -32.1502212]]


In [77]:
fc = gpkg.feature_classes['RS_setores_CD2022']
cursor = gpkg.connection.execute(f"""
select {fc.geometry_column_name}, id, CD_SETOR from {fc.name};
""")
rows = cursor.fetchall()
points_of_interest = [
    # ("inf", Point(-51.1204993, -30.0687244)), # predio inf
    ("reitoria", Point(-51.2201065, -30.033637)),  # reitoria
    # ("rodoviaria", Point(-51.2218492, -30.0230628)) # rodoviaria
]
found=[]
for row in rows:
    polygon = Polygon(row[0].rings[0].coordinates)

    
    for point in points_of_interest:
        if(polygon.contains(point[1])):
            print(f"Inside: {row[1]}")
            found.append((point[0], row))
            print(len(row[0].rings[0].coordinates))

Inside: 16817
41


In [78]:
pi = (
    "14159265358979323846264338327950288419716939937510"
    "58209749445923078164062862089986280348253421170679"
    "82148086513282306647093844609550582231725359408128"
    "48111745028410270193852110555964462294895493038196"
    "44288109756659334461284756482337867831652712019091"
    "45648566923460348610454326648213393607260249141273"
    "72458700660631558817488152092096282925409171536436"
    "78925903600113305305488204665213841469519415116094"
    "33057270365759591953092186117381932611793105118548"
    "07446237996274956735188575272489122793818301194912"
    "98336733624406566430860213949463952247371907021798"
    "60943702770539217176293176752384674818467669405132"
    "00056812714526356082778577134275778960917363717872"
    "14684409012249534301465495853710507922796892589235"
    "42019956112129021960864034418159813629774771309960"
    "51870721134999999837297804995105973173281609631859"
    "50244594553469083026425223082533446850352619311881"
    "71010003137838752886587533208381420617177669147303"
    "59825349042875546873115956286388235378759375195778"
    "18577805321712268066130019278766111959092164201989"
)
e = (
    "71828182845904523536028747135266249775724709369995"
    "95749669676277240766303535475945713821785251664274"
    "27466391932003059921817413596629043572900334295260"
    "59563073813232862794349076323382988075319525101901"
    "15738341879307021540891499348841675092447614606680"
    "82264800168477411853742345442437107539077744992069"
    "55170276183860626133138458300075204493382656029760"
    "67371132007093287091274437470472306969772093101416"
    "92836819025515108657463772111252389784425056953696"
    "77078544996996794686445490598793163688923009879312"
    "77361782154249992295763514822082698951936680331825"
    "28869398496465105820939239829488793320362509443117"
    "30123819706841614039701983767932068328237646480429"
    "53118023287825098194558153017567173613320698112509"
    "96181881593041690351598888519345807273866738589422"
    "87922849989208680582574927961048419844436346324496"
    "84875602336248270419786232090021609902353043699418"
    "49146314093431738143640546253152096183690888707016"
    "76839642437814059271456354906130310720851038375051"
    "01157477041718986106873969655212671546889570350354"
)
possible_profits = [int(f"{a}{b}") for a, b in zip(pi[::2], pi[1::2])]
possible_costs = [int(f"{a}{b}") for a, b in zip(e[::2], e[1::2])]
print(possible_profits)
print(possible_costs)

[14, 15, 92, 65, 35, 89, 79, 32, 38, 46, 26, 43, 38, 32, 79, 50, 28, 84, 19, 71, 69, 39, 93, 75, 10, 58, 20, 97, 49, 44, 59, 23, 7, 81, 64, 6, 28, 62, 8, 99, 86, 28, 3, 48, 25, 34, 21, 17, 6, 79, 82, 14, 80, 86, 51, 32, 82, 30, 66, 47, 9, 38, 44, 60, 95, 50, 58, 22, 31, 72, 53, 59, 40, 81, 28, 48, 11, 17, 45, 2, 84, 10, 27, 1, 93, 85, 21, 10, 55, 59, 64, 46, 22, 94, 89, 54, 93, 3, 81, 96, 44, 28, 81, 9, 75, 66, 59, 33, 44, 61, 28, 47, 56, 48, 23, 37, 86, 78, 31, 65, 27, 12, 1, 90, 91, 45, 64, 85, 66, 92, 34, 60, 34, 86, 10, 45, 43, 26, 64, 82, 13, 39, 36, 7, 26, 2, 49, 14, 12, 73, 72, 45, 87, 0, 66, 6, 31, 55, 88, 17, 48, 81, 52, 9, 20, 96, 28, 29, 25, 40, 91, 71, 53, 64, 36, 78, 92, 59, 3, 60, 1, 13, 30, 53, 5, 48, 82, 4, 66, 52, 13, 84, 14, 69, 51, 94, 15, 11, 60, 94, 33, 5, 72, 70, 36, 57, 59, 59, 19, 53, 9, 21, 86, 11, 73, 81, 93, 26, 11, 79, 31, 5, 11, 85, 48, 7, 44, 62, 37, 99, 62, 74, 95, 67, 35, 18, 85, 75, 27, 24, 89, 12, 27, 93, 81, 83, 1, 19, 49, 12, 98, 33, 67, 33, 62, 44, 

In [79]:
polygons = [(name, Polygon(row[0].rings[0].coordinates)) for name, row in found]

In [83]:
from pyrosm import OSM

osm = OSM("../data/sul-260930.osm.pbf")

In [ ]:
import numpy as np
from shapely.geometry import LineString

def create(polygon, name):
    poly_gdf = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")
    G = ox.graph_from_polygon(polygon, network_type='walk', truncate_by_edge=True)
    nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

    # 2. Reproject data to EPSG:3857 (Web Mercator) for plotting alignment
    edges_3857 = edges_gdf.to_crs(epsg=3857)
    poly_3857 = poly_gdf.to_crs(epsg=3857)

    def get_geometry(edge):
        if edge.geometry is None or edge.geometry.is_empty:
            print("found one")
            # row.name is the MultiIndex (u, v, key)
            u, v = edge.name[0], edge.name[1]
            
            # Get start (u) and end (v) node point geometries
            p_u = nodes_gdf.loc[u].geometry
            p_v = nodes_gdf.loc[v].geometry
            
            return LineString([p_u, p_v])
        return edge.geometry

    print({f"{a}" for a in edges_gdf['highway']})

    filtered_edges = edges_gdf[(edges_gdf['highway'] == 'residential') | (edges_gdf['highway'] == 'living_street')]

    sampled_points = []

    def sample_points_from_line(line, interval=10):
        return [line.interpolate(d) for d in np.arange(0, line.length, interval)]

    for _, row in filtered_edges.iterrows():
        pts = sample_points_from_line(get_geometry(row), interval=10)  # Every 30 meters
        sampled_points.extend(pts)


    # Create GeoDataFrame for the points in EPSG:3857
    points_gdf = gpd.GeoDataFrame(geometry=sampled_points, crs="EPSG:4326").drop_duplicates(keep='first').reset_index(drop=True)
    points_3857 = points_gdf.to_crs(epsg=3857)

    # 4. Create the Map Figure
    _, ax = plt.subplots(figsize=(10, 10))

    # Layer 1: Road routes
    edges_3857.plot(
        ax=ax, 
        linewidth=1.2, 
        edgecolor='#0066cc', 
        alpha=0.6, 
        zorder=2,
        label='Routes'
    )

    # Layer 2: Polygon boundary line
    poly_3857.boundary.plot(
        ax=ax, 
        color='#d9534f', 
        linewidth=2, 
        linestyle='--', 
        zorder=3,
        label='Boundary'
    )

    # Layer 3: OVERLAY SAMPLED POINTS
    points_3857.plot(
        ax=ax,
        color='#ff99ff',      # Bright orange for high contrast
        markersize=100,        # Point size
        edgecolor='black',    # Dark border around points for visibility
        linewidth=0.5,
        zorder=4,             # Higher zorder places points above routes
        label='Sampled Points'
    )

    # Layer 4: Background Map Tiles
    cx.add_basemap(
        ax,
        crs=poly_3857.crs.to_string(),
        source=cx.providers.Esri.WorldImagery
    )

    # Crop view to polygon bounds
    xmin, ymin, xmax, ymax = poly_3857.total_bounds
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    ax.set_axis_off()

    plt.savefig(f"../data/{name}_route_map_with_points.png", dpi=300, bbox_inches='tight', pad_inches=0.1)
    plt.show()
    # Save image
    plt.close()

    print("Map with overlaid points saved successfully.")

    points_gdf['profit'] = possible_profits[:len(points_gdf)]
    points_gdf['cost'] = possible_costs[:len(points_gdf)]

    ox.save_graphml(G, filepath=f"../{name}_vale.graphml")
    points_gdf.to_file(f'../{name}_sampled_points.gpkg', layer='route_points', driver='GPKG')

In [81]:
for name, polygon in polygons:
    create(polygon, name)

ConnectTimeout: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<HTTPSConnection(host='overpass-api.de', port=443) at 0x7f8220b97d70>, 'Connection to overpass-api.de timed out. (connect timeout=180)'))